# 05 - Baseline: CRNN + CTC trained from scratch

First OCR baseline, trained only on our own data (no pretrained weights), before comparing the starter pack's pretrained approaches (VLM/PaddleOCR/Kraken). Architecture: CNN feature extractor + bidirectional LSTM + CTC loss/decoding (Shi, Bai & Yao, 2015 - see RESOURCES.md), the standard approach for variable-length line-level text recognition. Diagram of the architecture: https://claude.ai/code/artifact/c0349616-105a-467d-87c3-e30a3df58d52

Trained on the full `fit` split (3,280 rows, ~24s/epoch on this GPU), evaluated on a fixed 120-row sample of the `val` split - the same sample size later used to compare PaddleOCR/Kraken/VLM, so results are comparable across approaches.

In [4]:
import sys
sys.path.append('..')

import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset


from src.config import RANDOM_STATE, MODELS_DIR
from src.data import build_image_path, load_raw_data, split_train_val
from src.evaluate import combined_score

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
train = load_raw_data('train')
fit_df, val_df = split_train_val(train)

# Fixed small validation sample - same size/seed will be reused to compare
# PaddleOCR/Kraken/VLM against this baseline on equal footing.
val_sample = val_df.sample(120, random_state=RANDOM_STATE).reset_index(drop=True)
len(fit_df), len(val_df), len(val_sample)

(3280, 818, 120)

In [3]:
# Character vocabulary from the fit split (index 0 reserved for the CTC blank token)
chars = sorted(set(''.join(fit_df['Target'].tolist())))
BLANK = 0
char_to_idx = {c: i + 1 for i, c in enumerate(chars)}
idx_to_char = {i + 1: c for i, c in enumerate(chars)}
NUM_CLASSES = len(chars) + 1
NUM_CLASSES

82

In [4]:
TARGET_HEIGHT = 32
MAX_WIDTH = 1024


def load_and_resize(record_id):
    img = Image.open(build_image_path(record_id)).convert('L')
    w, h = img.size
    new_w = max(8, min(MAX_WIDTH, round(w * TARGET_HEIGHT / h)))
    img = img.resize((new_w, TARGET_HEIGHT), Image.BILINEAR)
    arr = (np.asarray(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0)  # 1 x H x W

**CTC feasibility check.** CTC requires each sample's post-CNN time-step count `T` to be >= its target text length (the network needs at least one time step per character to emit it). With height=32 and the CNN's 4x width downsampling (below), 9 of 3,280 fit rows violate this - all the same kind of cramped multi-line/odd crops flagged during image EDA (`notebooks/02_eda_images.ipynb`), e.g. `PO7QQLWIFOT65BTz` and `EcxuqKeZl7OQexfB`. `nn.CTCLoss(zero_infinity=True)` zeroes the loss/gradient for these specific samples instead of producing NaN/inf, so they're silently skipped rather than corrupting the batch - acceptable at this rate (0.27%) for a baseline.

In [5]:
class LineDataset(Dataset):
    def __init__(self, df):
        self.ids = df['ID'].tolist()
        self.targets = df['Target'].tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = load_and_resize(self.ids[i])
        text = self.targets[i]
        label = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
        return img, label, text


def collate_fn(batch):
    imgs, labels, texts = zip(*batch)
    widths = [im.shape[-1] for im in imgs]
    max_w = max(widths)
    padded = torch.zeros(len(imgs), 1, TARGET_HEIGHT, max_w)
    for i, im in enumerate(imgs):
        padded[i, :, :, : im.shape[-1]] = im
    # Two width-halving maxpools in the CNN (conv1, conv2) -> total width stride 4.
    input_lengths = torch.tensor([(w // 2) // 2 for w in widths], dtype=torch.long)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return padded, torch.cat(labels), input_lengths, target_lengths, texts

In [6]:
class CRNN(nn.Module):
    """CNN + BiLSTM + CTC head (Shi, Bai & Yao 2015), see RESOURCES.md.
    Adds dropout (Srivastava et al., 2014, JMLR) vs. the original baseline:
    spatial dropout after the last two CNN blocks, inter-layer LSTM dropout,
    and dropout before the final classifier."""

    def __init__(self, num_classes, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d((2, 1)),
            nn.Dropout2d(dropout),
            nn.Conv2d(128, 256, (2, 1), 1, 0), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        feat = self.cnn(x)  # B x C x 1 x W'
        feat = feat.squeeze(2).permute(0, 2, 1)  # B x W' x C
        seq, _ = self.rnn(feat)
        seq = self.dropout(seq)
        return self.fc(seq).log_softmax(2)  # B x W' x num_classes


def greedy_decode(log_probs, input_lengths):
    """Argmax per time step, collapse repeats, drop blanks."""
    pred_idx = log_probs.argmax(2)
    texts = []
    for b in range(pred_idx.size(0)):
        seq = pred_idx[b, : input_lengths[b]].tolist()
        out, prev = [], None
        for idx in seq:
            if idx != prev and idx != BLANK:
                out.append(idx_to_char[idx])
            prev = idx
        texts.append(''.join(out))
    return texts

In [7]:
fit_ds = LineDataset(fit_df)
fit_dl = DataLoader(fit_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

val_ds = LineDataset(val_sample)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

DROPOUT = 0.2
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 50
PATIENCE = 8
RECORDED_BASELINE_SCORE = 0.568

model = CRNN(NUM_CLASSES, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)

In [17]:
import copy

def evaluate(loader):
    model.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for imgs, targets_cat, input_lengths, target_lengths, texts in loader:
            imgs = imgs.to(device)
            log_probs = model(imgs)
            preds = greedy_decode(log_probs.cpu(), input_lengths)
            refs.extend(texts)
            hyps.extend(preds)
    model.train()
    return combined_score(refs, hyps)


def train_with_early_stopping(model, optimizer, max_epochs, patience, initial_best_score=-1.0):
    """Trains until `max_epochs` or `patience` epochs pass with no val-score
    improvement (Prechelt 1998, see RESOURCES.md); returns the best checkpoint."""
    t0 = time.time()
    history = []
    best_score = initial_best_score
    best_epoch = 0  # 0 means no epoch beat initial_best_score
    best_state = copy.deepcopy(model.state_dict())
    epochs_since_improvement = 0

    for epoch in range(1, max_epochs + 1):
        total_loss = 0.0
        for imgs, targets_cat, input_lengths, target_lengths, texts in fit_dl:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            log_probs = model(imgs)
            loss = ctc_loss(log_probs.permute(1, 0, 2), targets_cat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        train_loss = total_loss / len(fit_ds)

        score = evaluate(val_dl)
        history.append({'epoch': epoch, 'train_loss': train_loss, **score})
        print(f"epoch {epoch:2d} | loss {train_loss:.3f} | val CER {score['cer_weighted']:.3f} "
              f"WER {score['wer_weighted']:.3f} final {score['score']:.3f} | {time.time()-t0:.0f}s")

        if score['score'] > best_score:
            best_score = score['score']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_since_improvement = 0
        else:
            epochs_since_improvement += 1
            if epochs_since_improvement >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    return best_state, best_score, best_epoch, history

In [ ]:
best_state, best_score, best_epoch, history = train_with_early_stopping(
    model, optimizer, max_epochs=MAX_EPOCHS, patience=PATIENCE
)

In [12]:
model.load_state_dict(best_state)
final_score = evaluate(val_dl)
print(f"best epoch: {best_epoch}")
print(f"regularized score: {final_score['score']:.3f} (baseline was {RECORDED_BASELINE_SCORE:.3f})")
print(f"delta: {final_score['score'] - RECORDED_BASELINE_SCORE:+.3f}")
final_score

best epoch: 50
regularized score: 0.691 (baseline was 0.568)
delta: +0.123


{'wer_weighted': 5.133333333333334,
 'cer_weighted': 10.416666666666666,
 'score': 0.6914141414141414}

In [13]:
# A few qualitative examples
model.eval()
with torch.no_grad():
    imgs, targets_cat, input_lengths, target_lengths, texts = next(iter(val_dl))
    imgs = imgs.to(device)
    preds = greedy_decode(model(imgs).cpu(), input_lengths)
for pred, ref in list(zip(preds, texts))[:8]:
    print(f"pred: {pred!r}")
    print(f"ref:  {ref!r}")
    print()

pred: 'near the Sowne of Sohechrautts in the Island of Barbadosnnt'
ref:  'near the Towne of St Michaells in the Island of Barbados next'

pred: 'personally appeared before me the withn named'
ref:  'personally appeared before me the within named'

pred: 'the peresones abovenantuined, ever have had heid or Endiene'
ref:  'the persones above mentioned, ever have had, or Enjoyed'

pred: 'other pte Witnesseth that the aforesd Major Portesgue for & in'
ref:  'other pte Witnesseth that the aforesd Major Fortesque for & in ~'

pred: 'us the said Capt Gorge Sthnpfast and Rolert prmings his attomeys peviall and'
ref:  'us the said Capt George Stanfast and Robert Jenning his attorney Genall and'

pred: 'To all Chrstian people to whom these pnts shall come'
ref:  'To all Christian people to whom these pnts shall come'

pred: 'releeye quint claymme and for ever tonfiormed and by theed prdseuts by'
ref:  'released quit clayme and for ever confirmed and by these by presents by-'

pred: 'who uson th

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if final_score['score'] > RECORDED_BASELINE_SCORE:
    model_path = MODELS_DIR / 'crnn_regularized.pt'
    torch.save({
        'model_state_dict': best_state,
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char,
        'target_height': TARGET_HEIGHT,
        'max_width': MAX_WIDTH,
    }, model_path)
    print(f"Saved new reference model: {model_path}")
else:
    print("No improvement over baseline (0.568) - keeping crnn_baseline.pt as reference. "
          "Log this as a negative result in README.md.")

Saved new reference model: C:\Users\alher\Desktop\handwritting_barbados\models\crnn_regularized.pt


In [15]:
CONTINUE_MAX_EPOCHS = 50
CONTINUE_PATIENCE = 8

checkpoint = torch.load(MODELS_DIR / 'crnn_regularized.pt')
model = CRNN(NUM_CLASSES, dropout=DROPOUT).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)

PREVIOUS_BEST_SCORE = evaluate(val_dl)['score']
PREVIOUS_BEST_SCORE

0.6914141414141414

In [18]:
best_state, best_score, best_epoch, history = train_with_early_stopping(
    model, optimizer, max_epochs=CONTINUE_MAX_EPOCHS, patience=CONTINUE_PATIENCE,
    initial_best_score=PREVIOUS_BEST_SCORE
)

epoch  1 | loss 0.320 | val CER 10.775 WER 5.358 final 0.679 | 27s
epoch  2 | loss 0.305 | val CER 10.667 WER 5.367 final 0.679 | 54s
epoch  3 | loss 0.292 | val CER 10.333 WER 5.167 final 0.691 | 82s
epoch  4 | loss 0.294 | val CER 10.425 WER 5.167 final 0.690 | 108s
epoch  5 | loss 0.289 | val CER 10.283 WER 5.175 final 0.691 | 135s
epoch  6 | loss 0.287 | val CER 10.233 WER 5.125 final 0.693 | 162s
epoch  7 | loss 0.277 | val CER 10.617 WER 5.425 final 0.677 | 187s
epoch  8 | loss 0.272 | val CER 10.508 WER 5.200 final 0.688 | 213s
epoch  9 | loss 0.269 | val CER 10.408 WER 5.192 final 0.689 | 239s
epoch 10 | loss 0.261 | val CER 10.000 WER 5.058 final 0.698 | 264s
epoch 11 | loss 0.259 | val CER 10.133 WER 5.092 final 0.696 | 291s
epoch 12 | loss 0.252 | val CER 10.300 WER 5.158 final 0.691 | 318s
epoch 13 | loss 0.250 | val CER 10.133 WER 5.108 final 0.695 | 344s
epoch 14 | loss 0.242 | val CER 10.150 WER 5.242 final 0.689 | 370s
epoch 15 | loss 0.240 | val CER 10.025 WER 5.133 fi

In [19]:
model.load_state_dict(best_state)
final_score = evaluate(val_dl)
print(f"best epoch in continuation (0 = no improvement): {best_epoch}")
print(f"score: {final_score['score']:.3f} (checkpoint score was {PREVIOUS_BEST_SCORE:.3f})")
print(f"delta: {final_score['score'] - PREVIOUS_BEST_SCORE:+.3f}")
final_score

best epoch in continuation (0 = no improvement): 18
score: 0.702 (checkpoint score was 0.691)
delta: +0.011


{'wer_weighted': 4.975, 'cer_weighted': 9.95, 'score': 0.7022537878787879}

In [20]:
if final_score['score'] > PREVIOUS_BEST_SCORE:
    model_path = MODELS_DIR / 'crnn_regularized.pt'
    torch.save({
        'model_state_dict': best_state,
        'char_to_idx': char_to_idx,
        'idx_to_char': idx_to_char,
        'target_height': TARGET_HEIGHT,
        'max_width': MAX_WIDTH,
    }, model_path)
    print(f"Updated reference model: {model_path}")
else:
    print(f"No improvement over {PREVIOUS_BEST_SCORE:.3f} - keeping existing crnn_regularized.pt. "
          "Log this as a negative result in README.md.")

Updated reference model: C:\Users\alher\Desktop\handwritting_barbados\models\crnn_regularized.pt
